# TASK-006 · 원본 baseline → Qwen3.5-9B

**범위: 데이터 점검 → 고정 분할 → 소량 GPU 검사 → 학습 전/후 기준 평가.**

출발 코드: `(260902)_baseline_desktop5060ti_offline(2).ipynb`.
원본의 `VQAMCDataset`, `DataCollator`, `DataLoader`, AdamW 학습 루프를 기반으로 수정했습니다.
이 파일은 작성·CPU 검사된 실행 코드이며, 작성 환경에서는 GPU 실행하지 않았습니다.

### 처음 실행
1. 노트북 옆 `data/train.csv`, `data/train/`을 준비하고 **설정 셀의 DATA_DIR**을 확인합니다.
2. 위에서 아래로 실행합니다. 첫 실행은 패키지와 모델을 다운로드합니다. Python 3.11/3.12 및 NVIDIA 드라이버가 필요합니다.
3. 단계별 실행 셀에서 멈출 수 있습니다. 한꺼번에 실행하면 데이터 점검부터 기준 비교까지 순차 수행합니다.
4. 다시 열면 설정·준비·정의 셀을 실행한 뒤 원하는 단계 셀을 실행합니다. 완료 결과는 검증 후 재사용합니다.

최종 확인 약 1,000개는 **분할만 보존**합니다. dev/test를 읽거나 제출 파일을 만들지 않습니다.
학습 1,000 / 튜닝 검증 약 500 / 최종 확인 약 1,000이 기본값입니다.

### 원본 대비 변경
| 구분 | 변경 내용 |
|---|---|
| 유지 | 프롬프트, Dataset→Collator→DataLoader, NF4 double quant, rank 8/alpha 16/dropout .05, LR 1e-4, AdamW, warmup 3%, 1 epoch, 누적 4, 384² |
| loss 유지 | 전체 입력을 labels로 복사. 인위적인 padding만 -100. **이미지·역할 특수 토큰도 원본처럼 감독** |
| 3.5 호환 | 모델/Transformers 버전, 모델 클래스, thinking OFF, BF16 compute/autocast 통일, 16GB용 메모리 절약 k-bit 준비 |
| 입력/학습 수정 | EXIF 방향+RGB, 중복 special token 삽입 방지, 마지막 불완전 누적 배치도 업데이트 |
| 평가 수정 | 생성 토큰만 디코딩, 파싱 실패는 빈 예측으로 오답 처리. 임의 a 및 추가 fallback 없음 |
| 실험 기반 | 이미지 그룹 분할, 생성 Accuracy, 소량 backward, 결과·자원·재로드 검사, 단계별 재실행 |

원본의 validation loss 대신 생성 Accuracy를 공통 평가 기준으로 사용합니다.
전체 입력 loss와 정답 전용 loss 비교는 후속 단계이며 여기서는 바꾸지 않습니다.
원본의 prepare_model_for_kbit_training 전체 FP32 확장 대신 동결·norm FP32·입력 gradient·checkpointing을 적용합니다.
이러한 호환/평가 수정과 새 분할 때문에 과거 점수와 직접 비교하지 않습니다.

모델: [Qwen/Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B), [공식 Transformers 문서](https://huggingface.co/docs/transformers/en/model_doc/qwen3_5).


## 1. 경로와 고정 실험 설정
수치를 바꾸면 별도 결과 폴더를 사용합니다. 처음 GPU 소량 검사 후, 학습 규모를 변경할 경우 본 기준 실험을 시작하기 전에 설정을 다시 확정하세요.

In [1]:
from pathlib import Path
import sys, os, json, hashlib, subprocess, time, uuid
MODEL_ID = 'Qwen/Qwen3.5-9B'
MODEL_KIND = 'qwen'
MODEL_SLUG = 'Qwen3_5_9B'

MODEL_REVISION = 'c202236235762e1c871ad0ccb60c8ee5ba337b9a'
MODEL_FILES = ['chat_template.jinja', 'config.json', 'merges.txt', 'model.safetensors-00001-of-00004.safetensors', 'model.safetensors-00002-of-00004.safetensors', 'model.safetensors-00003-of-00004.safetensors', 'model.safetensors-00004-of-00004.safetensors', 'model.safetensors.index.json', 'preprocessor_config.json', 'tokenizer.json', 'tokenizer_config.json', 'video_preprocessor_config.json', 'vocab.json']
TRANSFORMERS_VERSION = '5.8.0'
PEFT_VERSION = '0.18.1'
BASELINE_SOURCE_SHA256 = '501d86a66ab2ae6fe52497089a2c9ef682b00ad05e243d83bc78b128c78f92fe'
DATA_DIR = Path("data").resolve()  # train.csv와 train/의 상위 경로
PROJECT_DIR = Path.cwd().resolve()
SESSION_TAG = "baseline_qwen35_r1"  # 이미지/분할 정책을 바꾸면 새 이름
TRAIN_N = 1000
VALID_N = 500
HOLDOUT_N = 1000
SEED = 42
IMAGE_SIZE = 384
INSTALL_PACKAGES = True
DOWNLOAD_MODEL_FILES = True

if not (3,10) <= sys.version_info[:2] <= (3,13):
    raise RuntimeError("Python 3.10~3.13 필요. 권장 3.11/3.12")
if not (DATA_DIR / "train.csv").is_file():
    raise FileNotFoundError(f"DATA_DIR를 수정하세요: {DATA_DIR / 'train.csv'}")
MODEL_DIR = PROJECT_DIR / "downloads" / "models" / MODEL_SLUG / MODEL_REVISION
ENV_DIR = PROJECT_DIR / "downloads" / "envs" / "TASK006_baseline_qwen35"
ENV_PYTHON = ENV_DIR / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
SETUP_DIR = PROJECT_DIR / "output" / "TASK-006" / "setup"
SETUP_DIR.mkdir(parents=True,exist_ok=True)
RUN_DIR = SETUP_DIR  # 설치/다운로드 로그 위치. 실제 실험 경로는 코드 정의 뒤 확정
CFG = {
 "task_id":"TASK-006-R1", "code_version":"baseline-qwen35-v1.0",
 "baseline_source_sha256":BASELINE_SOURCE_SHA256,"session_tag":SESSION_TAG,
 "model_id":MODEL_ID,"revision":MODEL_REVISION,"kind":"qwen",
 "model_dir":str(MODEL_DIR),"data_dir":str(DATA_DIR),"seed":SEED,
 "train_n":TRAIN_N,"valid_n":VALID_N,"holdout_n":HOLDOUT_N,
 "near_hash_distance":4,"pixel_budget":IMAGE_SIZE**2,
 "epochs":1,"batch_size":1,"gradient_accumulation":4,"learning_rate":1e-4,
 "max_new_tokens":2,"max_input_tokens":4096,
 "quantization":"NF4/double-quant/BF16","loss":"baseline_full_sequence; padding_only_mask",
 "evaluation":"generated_only; strict_parse; failure_is_wrong; no_fallback",
 "image_policy":"EXIF/RGB; processor aspect-preserving pixel budget",
 "review_02":"pending","dev_used":False,"test_used":False,
}
print("데이터:", DATA_DIR)
print("학습/튜닝검증/최종검증 목표:",TRAIN_N,VALID_N,HOLDOUT_N)


데이터: C:\Users\SSAFY\Desktop\AI2_Challenge\data
학습/튜닝검증/최종검증 목표: 1000 500 1000


## 2. 전용 환경 설치
노트북 커널에 torch를 설치하지 않고 전용 Python에서 각 단계를 실행합니다. 설치 후 커널 재시작은 필요 없습니다. 인터넷은 설치와 다음 모델 다운로드에만 사용합니다.

In [2]:
import venv

def run_command(args, log_name):
    log_path=RUN_DIR/log_name
    env=os.environ.copy(); env["PYTHONUNBUFFERED"]="1"; env["PYTHONIOENCODING"]="utf-8"
    with open(log_path,"a",encoding="utf-8") as log:
        proc=subprocess.Popen([str(x) for x in args],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
            text=True,encoding="utf-8",errors="replace",env=env)
        try:
            for line in proc.stdout:
                print(line,end="");log.write(line);log.flush()
            if proc.wait()!=0: raise RuntimeError(f"실행 실패: {log_path}")
        except BaseException:
            if proc.poll() is None:
                proc.terminate()
                try:proc.wait(timeout=10)
                except subprocess.TimeoutExpired:proc.kill();proc.wait()
            raise


DEPENDENCIES = [
    "transformers==" + TRANSFORMERS_VERSION, "peft==" + PEFT_VERSION,
    "accelerate==1.12.0", "bitsandbytes==0.49.2", "pandas==2.3.3", "numpy==2.2.6",
    "pillow==12.1.0", "safetensors>=0.6.2,<1", "sentencepiece>=0.2,<1", "protobuf>=5,<7",
    "timm==1.0.24", "einops==0.8.1", "psutil>=6,<8", "tqdm>=4.67,<5",
]
INSTALL_SPEC = {"torch":"2.11.0", "torchvision":"0.26.0", "cuda_index":"cu128",
                "dependencies":DEPENDENCIES, "python":sys.version_info[:2]}
spec_text = json.dumps(INSTALL_SPEC, sort_keys=True)
spec_hash = hashlib.sha256(spec_text.encode()).hexdigest()
receipt = ENV_DIR / "install_spec.sha256"
if INSTALL_PACKAGES:
    if not ENV_PYTHON.exists():
        venv.EnvBuilder(with_pip=True).create(ENV_DIR)
    if not receipt.exists() or receipt.read_text().strip() != spec_hash:
        run_command([ENV_PYTHON, "-m", "pip", "install", "--upgrade", "pip"], "install.log")
        run_command([ENV_PYTHON, "-m", "pip", "install", "torch==2.11.0", "torchvision==0.26.0",
                     "--index-url", "https://download.pytorch.org/whl/cu128"], "install.log")
        run_command([ENV_PYTHON, "-m", "pip", "install", *DEPENDENCIES], "install.log")
        run_command([ENV_PYTHON, "-m", "pip", "check"], "install.log")
        receipt.write_text(spec_hash, encoding="utf-8")
    else:
        print("동일 설정의 전용 환경을 재사용합니다.")
elif not ENV_PYTHON.exists() or not receipt.exists() or receipt.read_text().strip() != spec_hash:
    raise RuntimeError("동일한 전용 실행 환경을 먼저 준비하세요. INSTALL_PACKAGES=True 필요.")

freeze = subprocess.check_output([str(ENV_PYTHON), "-m", "pip", "freeze"], text=True, encoding="utf-8")
(RUN_DIR / "requirements.lock.txt").write_text(freeze, encoding="utf-8")
(RUN_DIR / "install_spec.json").write_text(spec_text, encoding="utf-8")
print("실행 환경:", ENV_PYTHON)



  Using cached pip-26.2.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.2.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0
Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached torch-2.11.0%2Bcu128-cp311-cp311-win_amd64.whl.metadata (29 kB)
  Using cached torchvision-0.26.0%2Bcu128-cp311-cp311-win_amd64.whl.metadata (5.6 kB)
  Using cached filelock-3.32.3-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached pillow-12.3.0-cp311-cp311-win_amd64.whl.metada

## 3. Qwen3.5-9B 다운로드
고정 revision을 사용합니다. 파일 크기와 SHA256을 확인해 완료 파일을 재사용합니다. 중단된 파일은 다음 실행 시 다시 다운로드합니다. 이미 받은 모델이 있다면 설정의 MODEL_DIR을 해당 revision 폴더로 지정할 수 있습니다(다운로드 manifest 필요).

In [3]:
# 다운로드 전용: 모델 저장소의 고정 revision 파일을 HTTPS GET으로 받습니다.
# 외부 추론 API, HfApi, InferenceClient, snapshot_download, 사용자 데이터 업로드 없음.
import urllib.request
import urllib.parse
import urllib.error
import getpass
import shutil

class SafeRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        if urllib.parse.urlparse(newurl).scheme != "https":
            raise RuntimeError("HTTPS 이외의 다운로드 리디렉션을 거부합니다.")
        redirected = super().redirect_request(req, fp, code, msg, headers, newurl)
        if redirected is not None and urllib.parse.urlparse(newurl).hostname != "huggingface.co":
            redirected.remove_header("Authorization")
        return redirected

def hash_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda:f.read(4*1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

MODEL_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = MODEL_DIR / "download_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
records = manifest.get("files", {}) if manifest.get("revision") == MODEL_REVISION else {}
pending = []
for name in MODEL_FILES:
    p = MODEL_DIR / name
    record = records.get(name, {})
    valid = (p.is_file() and p.stat().st_size == record.get("size")
             and hash_file(p) == record.get("sha256"))
    if not valid:
        pending.append(name)
if pending and not DOWNLOAD_MODEL_FILES:
    raise FileNotFoundError("모델 파일 준비가 필요합니다: " + ", ".join(pending))

token = None
if pending and MODEL_KIND == "gemma":
    token = os.environ.get("HF_TOKEN") or getpass.getpass(
        "Gemma 사용 동의를 마친 Hugging Face read token (화면/파일에 저장하지 않음): ").strip()
    if not token:
        raise RuntimeError("Gemma 다운로드에는 모델 사용 동의와 read token이 필요합니다.")
opener = urllib.request.build_opener(SafeRedirect())
try:
    for name in pending:
        if Path(name).name != name:
            raise ValueError("예상하지 않은 모델 파일 경로")
        url = "https://huggingface.co/" + MODEL_ID + "/resolve/" + MODEL_REVISION + "/" + name
        headers = {"User-Agent":"local-vqa-asset-download/1.0"}
        if token:
            headers["Authorization"] = "Bearer " + token
        target = MODEL_DIR / name
        partial = target.with_name(target.name + ".part")
        print("다운로드:", name, flush=True)
        for attempt in range(3):
            try:
                request = urllib.request.Request(url, headers=headers)
                digest = hashlib.sha256()
                size = 0
                with opener.open(request, timeout=120) as response, open(partial, "wb") as f:
                    if "text/html" in response.headers.get("Content-Type", ""):
                        raise RuntimeError("파일 대신 HTML 응답을 받았습니다.")
                    expected = response.headers.get("Content-Length")
                    last_report = time.monotonic()
                    for chunk in iter(lambda:response.read(4*1024*1024), b""):
                        f.write(chunk)
                        digest.update(chunk)
                        size += len(chunk)
                        if time.monotonic()-last_report > 15:
                            print(f"  {name}: {size/2**30:.2f} GiB", flush=True)
                            last_report = time.monotonic()
                if size == 0 or (expected and size != int(expected)):
                    raise RuntimeError("다운로드 파일 길이 불일치")
                partial.replace(target)
                records[name] = {"size":size,"sha256":digest.hexdigest()}
                manifest_path.write_text(json.dumps({"model_id":MODEL_ID, "revision":MODEL_REVISION,
                                                    "files":records}, indent=2), encoding="utf-8")
                break
            except urllib.error.HTTPError as exc:
                if exc.code in (401,403):
                    raise RuntimeError("모델 다운로드 권한이 없습니다. 모델 사용 승인·read token을 확인하세요.") from None
                if attempt == 2:
                    raise RuntimeError(f"{name}: HTTP {exc.code} 다운로드 실패") from None
            except (OSError, RuntimeError) as exc:
                if attempt == 2:
                    raise RuntimeError(f"{name}: 다운로드 실패 ({type(exc).__name__}). 네트워크/디스크를 확인하세요.") from None
finally:
    token = None
    if "headers" in globals():
        headers.pop("Authorization", None)
    if "request" in globals():
        request.remove_header("Authorization")
shutil.copy2(manifest_path, RUN_DIR / "model_assets.json")
print("모델 파일 준비 완료. 이후 학습·추론은 네트워크 차단 상태로 실행합니다.")



모델 파일 준비 완료. 이후 학습·추론은 네트워크 차단 상태로 실행합니다.


## 4. 원본 baseline 기반 함수 정의
이 절은 GPU를 실행하지 않습니다. 다음 실행 셀에서 별도 프로세스를 시작합니다. 모델·데이터를 외부로 보내지 않습니다.

In [4]:
WORKER_PARTS = []

### 공통 함수·원본 프롬프트

In [5]:
WORKER_PARTS.append(r'''
# Original baseline structure: Dataset -> Collator -> DataLoader -> AdamW training.
import argparse, csv, gc, hashlib, json, math, os, random, re, sys, time, traceback
from pathlib import Path
from dataclasses import dataclass
from typing import Any


def write_json(path,obj):
    path=Path(path); tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),encoding='utf-8'); tmp.replace(path)


def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''): h.update(block)
    return h.hexdigest()


def block_network():
    os.environ.update(HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',HF_HUB_DISABLE_TELEMETRY='1',
        WANDB_DISABLED='true',TOKENIZERS_PARALLELISM='false',CUBLAS_WORKSPACE_CONFIG=':4096:8')
    def guard(event,args):
        if event in {'socket.connect','socket.getaddrinfo','socket.sendto'}: raise RuntimeError('로컬 실행 중 네트워크 접근 금지')
    sys.addaudithook(guard)


def read_table(path,columns):
    import pandas as pd
    df=pd.read_csv(path,dtype=str,keep_default_na=False)
    if set(columns)-set(df.columns): raise ValueError('필수 컬럼 누락')
    if df.empty or df.id.duplicated().any(): raise ValueError('빈 데이터 또는 중복 ID')
    for c in columns:
        if df[c].str.strip().eq('').any(): raise ValueError(f'빈 필수 항목: {c}')
    return df


def resolve_image(root,value):
    value=str(value).replace('\\','/')
    root=Path(root).resolve(); p=(root/value).resolve()
    if '://' in value or ':' in value or not p.is_relative_to(root) or not p.is_file(): raise FileNotFoundError(p)
    return p


SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)


def build_mc_prompt(question,a,b,c,d):
    return (f'{question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n'
            '정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.')


def parse_answer(text):
    # Baseline's implicit 'a' replaced by explicit failure. No fallback in this baseline.
    s=re.sub(r'^(?:answer|정답)\s*[:：]\s*','',str(text).strip(),flags=re.I)
    m=re.fullmatch(r'(?:\(([a-d])\)|([a-d]))[.。]?',s,flags=re.I)
    return (m.group(1) or m.group(2)).lower() if m else None



''')

### 원본 Dataset·Collator

In [6]:
WORKER_PARTS.append(r'''
from torch.utils.data import Dataset


class VQAMCDataset(Dataset):
    def __init__(self,df,processor,train=True,data_dir=None):
        self.df=df.reset_index(drop=True); self.processor=processor; self.train=train; self.data_dir=data_dir
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        from PIL import Image,ImageOps
        row=self.df.iloc[i]
        with Image.open(resolve_image(self.data_dir,row['path'])) as f:
            img=ImageOps.exif_transpose(f).convert('RGB')
        user_text=build_mc_prompt(*(str(row[k]) for k in ['question','a','b','c','d']))
        messages=[{'role':'system','content':[{'type':'text','text':SYSTEM_INSTRUCT}]},
                  {'role':'user','content':[{'type':'image','image':img},{'type':'text','text':user_text}]}]
        if self.train:
            messages.append({'role':'assistant','content':[{'type':'text','text':str(row['answer'])}]})
        return {'messages':messages,'image':img}


@dataclass
class DataCollator:
    processor: Any
    train: bool=True
    max_input_tokens: int=4096
    def __call__(self,batch):
        texts,images=[],[]
        for sample in batch:
            texts.append(self.processor.apply_chat_template(sample['messages'],tokenize=False,
                add_generation_prompt=not self.train,enable_thinking=False))
            images.append(sample['image'])
        enc=self.processor(text=texts,images=images,padding=True,return_tensors='pt',add_special_tokens=False)
        if enc['input_ids'].shape[1]>self.max_input_tokens: raise ValueError('입력 길이 초과: 자동 잘라내기 없음')
        if self.train:
            # Original baseline full-sequence labels retained, including media/role tokens.
            # Only artificial padding is ignored. Batch size 1 normally has no padding.
            enc['labels']=enc['input_ids'].clone()
            enc['labels'][enc['attention_mask']==0]=-100
        return enc



''')

### Qwen3.5 로딩·LoRA

In [7]:
WORKER_PARTS.append(r'''
class ModelAdapter:
    """Stage runner bridge; actual input/training uses baseline Dataset and Collator."""
    def __init__(self,cfg,out):
        import torch
        from transformers import AutoProcessor,AutoModelForImageTextToText,BitsAndBytesConfig
        self.cfg=cfg;self.out=out; self.dtype=torch.bfloat16;self.device='cuda:0'
        if not torch.cuda.is_bf16_supported(): raise RuntimeError('BF16 지원 GPU 필요')
        self.processor=AutoProcessor.from_pretrained(cfg['model_dir'],local_files_only=True)
        ip=self.processor.image_processor; pixels=cfg['pixel_budget']
        ip.size={'shortest_edge':pixels,'longest_edge':pixels}
        if hasattr(ip,'min_pixels'): ip.min_pixels=pixels
        if hasattr(ip,'max_pixels'): ip.max_pixels=pixels
        self.tokenizer=self.processor.tokenizer
        bnb_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=self.dtype)
        self.model=AutoModelForImageTextToText.from_pretrained(cfg['model_dir'],
            local_files_only=True,quantization_config=bnb_config,device_map={'':'cuda:0'},
            dtype=self.dtype,attn_implementation='sdpa')
        self.model.config.use_cache=False; self.model.eval()
        self.processor.save_pretrained(out/'processor')
        write_json(out/'model_config.json',self.model.config.to_dict())
        write_json(out/'processor_settings.json',ip.to_dict())
    def add_lora(self):
        import torch
        from peft import LoraConfig,get_peft_model
        # Memory-safe k-bit preparation: avoid full embedding FP32 expansion on 16 GB.
        for name,p in self.model.named_parameters():
            p.requires_grad_(False)
            if 'norm' in name.lower() and p.ndim==1 and p.is_floating_point(): p.data=p.data.to(torch.float32)
        self.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
        self.model.enable_input_require_grads()
        lora_config=LoraConfig(r=8,lora_alpha=16,lora_dropout=.05,bias='none',
            target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
            task_type='CAUSAL_LM')
        self.model=get_peft_model(self.model,lora_config)
        self.model.print_trainable_parameters()
        write_json(self.out/'lora_parameters.json',[{'name':n,'shape':list(p.shape)} for n,p in self.model.named_parameters() if p.requires_grad])
    def move(self,batch):
        return {k:v.to(self.device,dtype=self.dtype if v.is_floating_point() else v.dtype) for k,v in batch.items()}
    def encode(self,row,training=False):
        import pandas as pd
        ds=VQAMCDataset(pd.DataFrame([row]),self.processor,training,self.cfg['data_dir'])
        sample=ds[0]; enc=DataCollator(self.processor,training,self.cfg['max_input_tokens'])([sample])
        grid=enc.get('image_grid_thw'); gs=grid.tolist() if grid is not None else []
        ip=self.processor.image_processor; patch=getattr(ip,'patch_size',16); merge=getattr(ip,'merge_size',2)
        self.last_image_meta={'original_size':list(sample['image'].size),'grid_thw':gs,
            'processed_hw':[[int(g[1])*patch,int(g[2])*patch] for g in gs],
            'visual_tokens':sum(math.prod(g)//(merge*merge) for g in gs)}
        return self.move(enc)
    def loss(self,inputs): return self.model(**inputs,use_cache=False).loss
    def generate(self,inputs):
        import torch
        self.model.eval()
        with torch.inference_mode(),torch.autocast('cuda',dtype=self.dtype):
            ids=self.model.generate(**inputs,max_new_tokens=self.cfg['max_new_tokens'],do_sample=False,
                use_cache=True,eos_token_id=self.tokenizer.eos_token_id)
        # Original baseline decoded the whole input; now decode generated tokens only.
        return self.tokenizer.decode(ids[0,inputs['input_ids'].shape[1]:],skip_special_tokens=True).strip()
    def reload_adapter(self,path):
        from peft import PeftModel
        base=self.model.unload()
        self.model=PeftModel.from_pretrained(base,str(path),local_files_only=True,is_trainable=False)
        self.model.eval()



''')

### 생성 평가

In [8]:
WORKER_PARTS.append(r'''
def evaluate(adapter,df,label,out):
    import torch,pandas as pd
    adapter.model.eval();torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats();start=time.perf_counter()
    rows=[]
    fields=['id','answer','raw_output','parse_failed','fallback_used','gold','correct','strict_correct','group_id','question_type','image_meta']
    with open(out/f'{label}_predictions.csv','w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader()
        for i,row in enumerate(df.to_dict('records')):
            inputs=adapter.encode({k:v for k,v in row.items() if k!='answer'})
            raw=adapter.generate(inputs);answer=parse_answer(raw)
            rec={'id':row['id'],'answer':answer or '', 'raw_output':raw,'parse_failed':answer is None,
                 'fallback_used':False,'gold':row['answer'],'correct':answer==row['answer'],
                 'strict_correct':answer==row['answer'],'group_id':row['group_id'],'question_type':row['question_type'],
                 'image_meta':json.dumps(adapter.last_image_meta)}
            writer.writerow(rec);f.flush();rows.append(rec);del inputs
            if (i+1)%25==0 or i+1==len(df): print(label,i+1,'/',len(df),flush=True)
    torch.cuda.synchronize();seconds=time.perf_counter()-start;n=len(rows)
    metrics={'n':n,'correct_n':sum(r['correct'] for r in rows),'accuracy':sum(r['correct'] for r in rows)/n,
        'parse_failure_rate':sum(r['parse_failed'] for r in rows)/n,'fallback_usage_rate':0.,
        'seconds':seconds,'seconds_per_sample':seconds/n,'peak_allocated_gib':torch.cuda.max_memory_allocated()/2**30,
        'peak_reserved_gib':torch.cuda.max_memory_reserved()/2**30}
    write_json(out/f'{label}_metrics.json',metrics);print(metrics,flush=True)
    return rows,metrics



''')

### 원본 DataLoader 기반 학습 루프

In [9]:
WORKER_PARTS.append(r'''
def train_one_epoch(adapter,df,cfg,out):
    import torch,pandas as pd
    from torch.utils.data import DataLoader
    from transformers import get_linear_schedule_with_warmup
    # Baseline DataLoader, AdamW, warmup, accumulation, one epoch retained.
    train_ds=VQAMCDataset(df,adapter.processor,train=True,data_dir=cfg['data_dir'])
    train_loader=DataLoader(train_ds,batch_size=1,shuffle=True,
        generator=torch.Generator().manual_seed(cfg['seed']),
        collate_fn=DataCollator(adapter.processor,True,cfg['max_input_tokens']),num_workers=0)
    model=adapter.model; GRAD_ACCUM=cfg['gradient_accumulation']
    params=[p for p in model.parameters() if p.requires_grad]
    optimizer=torch.optim.AdamW(params,lr=cfg['learning_rate'])
    num_training_steps=math.ceil(len(train_loader)/GRAD_ACCUM)
    scheduler=get_linear_schedule_with_warmup(optimizer,int(num_training_steps*.03),num_training_steps)
    # BF16 needs no GradScaler; original mixed FP16 compute/BF16 autocast is unified.
    model.train(); optimizer.zero_grad(set_to_none=True)
    torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats();started=time.perf_counter();rows=[];running=0.
    for step,batch in enumerate(train_loader,start=1):
        batch=adapter.move(batch)
        # Correct denominator and optimizer step for the final incomplete group.
        group_start=((step-1)//GRAD_ACCUM)*GRAD_ACCUM
        group_size=min(GRAD_ACCUM,len(train_loader)-group_start)
        with torch.autocast('cuda',dtype=adapter.dtype):
            outputs=model(**batch,use_cache=False);loss=outputs.loss
        if not torch.isfinite(loss): raise FloatingPointError('loss NaN/Inf')
        raw_loss=float(loss.detach());(loss/group_size).backward();running+=raw_loss
        del outputs,loss,batch
        if step%GRAD_ACCUM==0 or step==len(train_loader):
            grads=[p.grad for p in params if p.grad is not None]
            if not grads or not all(bool(torch.isfinite(g).all()) for g in grads): raise FloatingPointError('gradient 오류')
            optimizer.step();optimizer.zero_grad(set_to_none=True);scheduler.step()
            rec={'update':len(rows)+1,'mean_loss':running/group_size,'lr':scheduler.get_last_lr()[0]};rows.append(rec);running=0.
            with open(out/'train_updates.jsonl','a',encoding='utf-8') as f:f.write(json.dumps(rec)+'\n')
            print('train',rec['update'],'/',num_training_steps,rec,flush=True)
    torch.cuda.synchronize()
    metrics={'train_n':len(df),'epochs':1,'updates':len(rows),'seconds':time.perf_counter()-started,
        'peak_allocated_gib':torch.cuda.max_memory_allocated()/2**30,'peak_reserved_gib':torch.cuda.max_memory_reserved()/2**30}
    pd.DataFrame(rows).to_csv(out/'train_log.csv',index=False);write_json(out/'train_metrics.json',metrics)
    del optimizer,scheduler,params,grads,train_loader;gc.collect();torch.cuda.empty_cache()
    return metrics

''')

### 이미지 검사·분할·단계 제어
동일 RGB 픽셀은 같은 그룹으로 묶습니다. dHash 거리 ≤4이며 종횡비가 유사한 이미지도 보수적으로 같은 그룹에 묶고 후보 목록을 저장합니다. **유사도 휴리스틱은 중복을 모두 탐지하거나 확정하지 못합니다.** 후보와 분포를 검토해야 합니다. 문항 유형도 정규식 진단 태그이며 정답 라벨이 아닙니다.

그룹을 쪼개지 않기 때문에 실제 개수는 목표와 다를 수 있습니다. 검토 전 결과는 탐색용이며 02 검토 완료로 표시하지 않습니다.

In [10]:
WORKER_PARTS.append(r'''
def question_type(text):
    # Heuristic diagnostic tags, not verified semantic labels.
    if re.search(r'아닌|않은|않는|없는|제외', text): return 'negation'
    if re.search(r'가격|얼마|원인가|금액|할인', text): return 'price'
    if re.search(r'왼쪽|오른쪽|위쪽|아래|옆|위치', text): return 'position'
    if re.search(r'몇|숫자|날짜|시간|번호|수량', text): return 'number'
    return 'text_other'


def take_groups(frame, target, seed):
    import numpy as np
    # Never split a group. Greedy size selection over repeated seeded permutations;
    # use answer/type proportions to break ties and minimize distribution shift.
    groups = frame.groupby('group_id', sort=True).size()
    if target >= len(frame): return set(groups.index)
    rng = np.random.default_rng(seed)
    best, best_score = None, float('inf')
    answer_ref = frame.answer.value_counts(normalize=True)
    type_ref = frame.question_type.value_counts(normalize=True)
    for _ in range(100):
        chosen, n = [], 0
        for g in rng.permutation(groups.index.to_numpy()):
            size = int(groups[g])
            if abs(n + size - target) < abs(n-target):
                chosen.append(g); n += size
        sub = frame[frame.group_id.isin(chosen)]
        if sub.empty: continue
        def distance(col, ref):
            return (sub[col].value_counts(normalize=True).reindex(ref.index,fill_value=0)-ref).abs().sum()
        score = abs(n-target) + .1 * (distance('answer',answer_ref)+distance('question_type',type_ref))
        if score < best_score: best, best_score = set(chosen), score
    if best is None: raise ValueError('그룹 크기로 인해 분할 불가')
    return best


def audit(cfg, out):
    import pandas as pd
    import numpy as np
    from PIL import Image, ImageOps
    root = Path(cfg['data_dir'])
    df = read_table(root/'train.csv', ['id','path','question','a','b','c','d','answer']).sort_values('id').reset_index(drop=True)
    if not df.answer.isin(list('abcd')).all(): raise ValueError('라벨은 소문자 a~d여야 합니다.')
    findings = []
    for row in df.to_dict('records'):
        for col in ['question','a','b','c','d']:
            if row[col] != row[col].strip(): findings.append({'id':row['id'],'field':col,'reason':'leading/trailing whitespace','action':'report only'})
            if '\ufffd' in row[col]: findings.append({'id':row['id'],'field':col,'reason':'replacement character','action':'review'})
        if len({row[c].strip() for c in 'abcd'}) < 4:
            findings.append({'id':row['id'],'field':'options','reason':'duplicate options','action':'review'})
    pd.DataFrame(findings,columns=['id','field','reason','action']).to_csv(out/'data_findings.csv',index=False)
    images, errors = [], []
    for i,row in enumerate(df.to_dict('records')):
        try:
            p = resolve_image(root,row['path'])
            with Image.open(p) as raw:
                raw.load()
                orientation = raw.getexif().get(274,1)
                im = ImageOps.exif_transpose(raw).convert('RGB')
                pixel_hash = hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest()
                tiny=np.asarray(im.convert('L').resize((9,8),Image.Resampling.LANCZOS))
                bits=(tiny[:,1:]>tiny[:,:-1]).reshape(-1)
                dh=sum(int(v)<<j for j,v in enumerate(bits))
                stat=p.stat()
                images.append(dict(id=row['id'],path=row['path'],width=im.width,height=im.height,
                    orientation=orientation,pixel_sha256=pixel_hash,dhash=f'{dh:016x}',
                    file_sha256=sha256(p),bytes=stat.st_size,mtime_ns=stat.st_mtime_ns))
        except Exception as exc: errors.append(dict(id=row['id'],path=row['path'],error=str(exc)))
        if (i+1)%250==0: print('image audit',i+1,'/',len(df),flush=True)
    pd.DataFrame(errors,columns=['id','path','error']).to_csv(out/'image_errors.csv',index=False)
    if errors: raise RuntimeError(f'누락/손상 이미지 {len(errors)}개. image_errors.csv 확인. 자동 제외하지 않습니다.')
    parent=list(range(len(images)))
    def find(i):
        while parent[i]!=i: parent[i]=parent[parent[i]]; i=parent[i]
        return i
    def union(i,j):
        a,b=find(i),find(j)
        if a!=b: parent[max(a,b)]=min(a,b)
    exact={}; candidates=[]; hashes=[int(x['dhash'],16) for x in images]
    for i,row in enumerate(images):
        h=row['pixel_sha256']
        if h in exact: union(i,exact[h])
        else: exact[h]=i
        ratio=row['width']/row['height']
        for j in range(i):
            if images[j]['pixel_sha256']==h: continue
            distance=(hashes[i]^hashes[j]).bit_count()
            if distance<=cfg['near_hash_distance'] and abs(math.log(ratio/(images[j]['width']/images[j]['height'])))<=.05:
                # Conservative candidate grouping; false positives can reduce train pool.
                union(i,j)
                candidates.append({'id_a':images[j]['id'],'id_b':row['id'],'distance':distance,
                                   'policy':'same group conservatively; not human verified'})
    df['group_id']=['g_'+images[find(i)]['id'] for i in range(len(images))]
    df['question_type']=df.question.map(question_type)
    for i,row in enumerate(images): row['group_id']=df.iloc[i].group_id
    pd.DataFrame(images).to_csv(out/'image_audit.csv',index=False)
    pd.DataFrame(candidates,columns=['id_a','id_b','distance','policy']).to_csv(out/'near_duplicate_candidates.csv',index=False)
    if cfg['train_n']+cfg['valid_n']+cfg['holdout_n']>=len(df):
        raise ValueError('데이터 개수보다 분할 요청이 큽니다. config 값을 확인하세요.')
    final=take_groups(df,cfg['holdout_n'],cfg['seed'])
    remaining=df[~df.group_id.isin(final)]
    valid=take_groups(remaining,cfg['valid_n'],cfg['seed']+1)
    remaining=remaining[~remaining.group_id.isin(valid)]
    train=take_groups(remaining,cfg['train_n'],cfg['seed']+2)
    df['split']=['holdout' if g in final else 'valid' if g in valid else 'train' if g in train else 'pool' for g in df.group_id]
    if not set(['train','valid','holdout']).issubset(set(df.split)): raise ValueError('빈 분할이 있습니다.')
    assert df.groupby('group_id').split.nunique().max()==1
    split=df[['id','split','group_id','question_type']]
    split.to_csv(out/'split_manifest.csv',index=False)
    for name in ['train','valid','holdout','pool']:
        split[split.split==name].to_csv(out/f'{name}_ids.csv',index=False)
    dist=pd.crosstab(df.split,df.answer).reindex(columns=list('abcd'),fill_value=0)
    dist.to_csv(out/'answer_distribution.csv')
    pd.crosstab(df.split,df.question_type).to_csv(out/'type_distribution.csv')
    manifest={'csv_sha256':sha256(root/'train.csv'),'split_sha256':sha256(out/'split_manifest.csv'),
              'image_audit_sha256':sha256(out/'image_audit.csv'),'counts':df.split.value_counts().to_dict(),
              'groups':int(df.group_id.nunique()),'near_candidate_pairs':len(candidates),
              'near_policy':'dHash candidate transitive grouping; human review pending; misses possible',
              'question_type_policy':'regex heuristic only','review_02':'pending','dev_used':False,
              'test_used':False,'findings':len(findings)}
    write_json(out/'data_manifest.json',manifest)
    print(dist.to_string(),flush=True); print(json.dumps(manifest,ensure_ascii=False,indent=2),flush=True)
    return manifest


def completed(root, stage):
    p=root/f'{stage}_complete.json'
    if not p.exists(): return None
    rec=json.loads(p.read_text(encoding='utf-8'))
    for name,digest in rec['artifacts'].items():
        path=Path(rec['directory'])/name
        if not path.is_file() or sha256(path)!=digest: raise RuntimeError(f'완료 결과가 수정/누락되었습니다: {path}')
    return rec


def load_split(cfg, root):
    import pandas as pd
    record=completed(root,'audit')
    if record is None: raise RuntimeError('audit 단계를 먼저 실행하세요.')
    path=Path(record['directory'])
    meta=json.loads((path/'data_manifest.json').read_text(encoding='utf-8'))
    if sha256(Path(cfg['data_dir'])/'train.csv')!=meta['csv_sha256']: raise RuntimeError('train.csv 변경: 새 SESSION_TAG로 분할부터 실행하세요.')
    # Validate image bytes: changed data cannot silently reuse completed experiments.
    images=pd.read_csv(path/'image_audit.csv',dtype=str)
    for r in images.to_dict('records'):
        p=resolve_image(cfg['data_dir'],r['path'])
        if sha256(p)!=r['file_sha256']: raise RuntimeError(f"이미지 변경: {r['id']}. 새 SESSION_TAG 필요")
    df=pd.read_csv(Path(cfg['data_dir'])/'train.csv',dtype=str,keep_default_na=False)
    split=pd.read_csv(path/'split_manifest.csv',dtype=str)
    merged=df.merge(split,on='id',validate='one_to_one')
    assert len(merged)==len(df) and merged.groupby('group_id').split.nunique().max()==1
    return merged[merged.split=='train'].copy(),merged[merged.split=='valid'].copy(),meta


def gpu_environment(cfg,out):
    import torch, numpy as np, platform
    from importlib.metadata import version
    if not torch.cuda.is_available(): raise RuntimeError('CUDA GPU를 찾을 수 없습니다. 드라이버/설치를 확인하세요.')
    random.seed(cfg['seed']); np.random.seed(cfg['seed']); torch.manual_seed(cfg['seed']); torch.cuda.manual_seed_all(cfg['seed'])
    torch.backends.cudnn.benchmark=False
    torch.backends.cuda.matmul.allow_tf32=False
    torch.use_deterministic_algorithms(True,warn_only=True)
    x=torch.ones((32,32),device='cuda',dtype=torch.bfloat16)
    assert float((x@x)[0,0])==32.; del x
    info={'python':sys.version,'os':platform.platform(),'torch':torch.__version__,'cuda':torch.version.cuda,
          'gpu':torch.cuda.get_device_name(),'vram_gib':torch.cuda.get_device_properties(0).total_memory/2**30,
          'packages':{p:version(p) for p in ['transformers','peft','bitsandbytes','accelerate']},
          'network':'blocked; local model files only'}
    write_json(out/'environment.json',info); print(info,flush=True)


def smoke(adapter, tr, cfg, out):
    import torch
    row=tr.iloc[0].to_dict()
    clean={k:v for k,v in row.items() if k!='answer'}
    before=adapter.generate(adapter.encode(clean))
    adapter.add_lora(); adapter.model.train(); torch.cuda.reset_peak_memory_stats()
    start=time.perf_counter(); inputs=adapter.encode(row,training=True)
    if 'pixel_values' not in inputs or inputs['pixel_values'].numel()==0: raise RuntimeError('이미지 입력이 없습니다.')
    ids=inputs['input_ids'][0].tolist(); labels=inputs['labels'][0].tolist()
    import pandas as pd
    pd.DataFrame([{'position':i,'token_id':t,'token':adapter.tokenizer.convert_ids_to_tokens(t),
                   'supervised':labels[i]!=-100} for i,t in enumerate(ids)]).to_csv(out/'supervised_tokens.csv',index=False)
    assert sum(x!=-100 for x in labels)>0
    with torch.autocast('cuda',dtype=adapter.dtype): loss=adapter.loss(inputs)
    if not torch.isfinite(loss): raise FloatingPointError('유한하지 않은 loss')
    loss.backward()
    grads=[p.grad for p in adapter.model.parameters() if p.requires_grad and p.grad is not None]
    assert grads and all(bool(torch.isfinite(g).all()) for g in grads) and any(bool(g.abs().max()>0) for g in grads)
    torch.cuda.synchronize()
    result={'id':row['id'],'loss':float(loss.detach()),'generation':before,'parsed':parse_answer(before),
            'finite_nonzero_gradients':True,'optimizer_step_performed':False,
            'seconds_one_forward_backward':time.perf_counter()-start,
            'peak_allocated_gib':torch.cuda.max_memory_allocated()/2**30,
            'supervised_tokens':sum(x!=-100 for x in labels),'image_meta':adapter.last_image_meta,
            'note':'smoke only; runtime estimate is rough, not measured epoch duration'}
    adapter.model.zero_grad(set_to_none=True); write_json(out/'smoke_metrics.json',result)
    print(result,flush=True)
    return result


def compare(root, out):
    import pandas as pd
    b=completed(root,'base'); l=completed(root,'lora_eval')
    if b is None or l is None: raise RuntimeError('base와 lora_eval을 먼저 완료하세요.')
    a=pd.read_csv(Path(b['directory'])/'valid_base_predictions.csv',keep_default_na=False)
    z=pd.read_csv(Path(l['directory'])/'valid_lora_predictions.csv',keep_default_na=False)
    m=a.merge(z,on='id',suffixes=('_base','_lora'),validate='one_to_one')
    assert len(m)==len(a)==len(z) and (m.gold_base==m.gold_lora).all()
    good_a=m.answer_base==m.gold_base; good_z=m.answer_lora==m.gold_lora
    m['transition']=['gain' if not x and y else 'loss' if x and not y else 'same_correct' if x else 'same_wrong' for x,y in zip(good_a,good_z)]
    m.to_csv(out/'paired_predictions.csv',index=False)
    m[m.transition.isin(['gain','loss'])].to_csv(out/'changed_answers.csv',index=False)
    errors=m[~good_z].copy(); errors['manual_error_category']=''; errors['review_note']=''
    errors.to_csv(out/'error_review_template.csv',index=False)
    rows=[]
    for label,rec in [('base',b),('lora',l)]:
        metrics=json.loads((Path(rec['directory'])/f'valid_{label}_metrics.json').read_text())
        rows.append({'model':label,**metrics})
    pd.DataFrame(rows).to_csv(out/'comparison.csv',index=False)
    types=m.groupby('question_type_base').apply(lambda g:pd.Series({'n':len(g),
        'base_accuracy':(g.answer_base==g.gold_base).mean(),'lora_accuracy':(g.answer_lora==g.gold_lora).mean()}),include_groups=False)
    types.to_csv(out/'type_comparison.csv')
    result={'gain':int((~good_a & good_z).sum()),'loss':int((good_a & ~good_z).sum()),
            'accuracy_delta':float(good_z.mean()-good_a.mean()),'n':len(m),
            'adoption':'pending; no automatic model selection','review_02':'pending',
            'holdout_evaluated':False,'test_evaluated':False}
    write_json(out/'summary.json',result)
    (out/'PROJECT_STATUS_update.md').write_text('# TASK-006-R1 반영 후보\n\n기준 평가 실행 완료. 독립 검토 미완료.\n\n'+json.dumps(result,ensure_ascii=False,indent=2)+'\n\n실제 파일 위치: '+str(root),encoding='utf-8')
    (out/'CHANGELOG.md').write_text('# TASK-006-R1 실행 기록\n\n고정 분할에서 base/새 LoRA 평가 완료. 원본/최고 모델/공식 PROJECT_STATUS 수정 없음. 채택·제출·02 검토 미완료.\n',encoding='utf-8')
    print(pd.DataFrame(rows).to_string(index=False),flush=True);print(result,flush=True)
    return result


def main(cfg,stage):
    block_network()
    root=Path(cfg['run_dir']); root.mkdir(parents=True,exist_ok=True)
    # A fresh process per stage frees all CUDA memory when it exits.
    if stage!='audit': tr,va,data_info=load_split(cfg,root)
    old=completed(root,stage)
    if old:
        print('완료 단계 재사용:',stage,old['directory'],flush=True);return
    import uuid
    out=root/(stage+'_'+time.strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]);out.mkdir()
    write_json(out/'config.json',cfg)
    write_json(root/f'{stage}_status.json',{'state':'running','directory':str(out)})
    try:
        if stage=='audit': result=audit(cfg,out)
        elif stage=='compare': result=compare(root,out)
        else:
            if stage in ['base','train','lora_eval'] and completed(root,'smoke') is None:
                raise RuntimeError('smoke 검사를 먼저 완료하세요.')
            if stage=='lora_eval' and completed(root,'train') is None: raise RuntimeError('train 먼저 실행 필요')
            gpu_environment(cfg,out)
            adapter=ModelAdapter(cfg,out)
            if stage=='smoke': result=smoke(adapter,tr,cfg,out)
            elif stage=='base': _,result=evaluate(adapter,va,'valid_base',out)
            elif stage=='train':
                adapter.add_lora()
                result=train_one_epoch(adapter,tr,cfg,out)
                checkpoint=out/'adapter_epoch1'; adapter.model.save_pretrained(checkpoint);adapter.processor.save_pretrained(checkpoint)
                # Compare same few validation predictions immediately before/after disk reload.
                probe=va.head(min(5,len(va)))
                _,p=evaluate(adapter,probe,'reload_before',out)
                adapter.reload_adapter(checkpoint)
                _,q=evaluate(adapter,probe,'reload_after',out)
                import pandas as pd
                a=pd.read_csv(out/'reload_before_predictions.csv',keep_default_na=False)
                b=pd.read_csv(out/'reload_after_predictions.csv',keep_default_na=False)
                same=a[['id','raw_output','answer']].equals(b[['id','raw_output','answer']])
                write_json(out/'reload_check.json',{'n':len(probe),'identical_outputs':same})
                if not same: raise RuntimeError('저장 전/후 예측 불일치. 로그를 검토하세요.')
                result['checkpoint']=str(checkpoint)
            elif stage=='lora_eval':
                from peft import PeftModel
                train_rec=completed(root,'train'); checkpoint=Path(train_rec['directory'])/'adapter_epoch1'
                adapter.model=PeftModel.from_pretrained(adapter.model,str(checkpoint),local_files_only=True,is_trainable=False)
                adapter.base=adapter.model.get_base_model()
                _,result=evaluate(adapter,va,'valid_lora',out)
            else: raise ValueError(stage)
        files={str(p.relative_to(out)):sha256(p) for p in out.rglob('*') if p.is_file()}
        rec={'stage':stage,'directory':str(out),'result':result,'artifacts':files}
        write_json(root/f'{stage}_complete.json',rec)
        write_json(root/f'{stage}_status.json',{'state':'completed','directory':str(out)})
    except BaseException as exc:
        write_json(root/f'{stage}_status.json',{'state':'failed_or_interrupted','directory':str(out),'error':str(exc)})
        raise


if __name__=='__main__':
    cfg=json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))
    main(cfg,sys.argv[2])

''')

## 5. 실험 폴더 확정
코드·설정·CSV·설치 버전 해시로 독립 폴더를 만듭니다. 기존 실험은 덮어쓰지 않습니다. 한 폴더에서 동시에 두 작업을 실행하면 잠금으로 중단합니다.

In [11]:
source="\n\n".join(WORKER_PARTS)
compile(source,"task006_worker.py","exec")
CFG["worker_sha256"]=hashlib.sha256(source.encode()).hexdigest()
CFG["train_csv_sha256"]=hash_file(DATA_DIR/"train.csv")
CFG["environment_lock_sha256"]=hash_file(SETUP_DIR/"requirements.lock.txt")
CFG["model_assets_sha256"]=hash_file(SETUP_DIR/"model_assets.json")
fingerprint=hashlib.sha256(json.dumps(CFG,sort_keys=True).encode()).hexdigest()[:16]
EXPERIMENT_ID="TASK006-"+fingerprint
RUN_DIR=PROJECT_DIR/"output"/"TASK-006"/EXPERIMENT_ID
RUN_DIR.mkdir(parents=True,exist_ok=True)
CFG["run_dir"]=str(RUN_DIR); CFG["experiment_id"]=EXPERIMENT_ID
worker=RUN_DIR/"task006_worker.py"
worker.write_text(source,encoding="utf-8")
(RUN_DIR/"run_config.json").write_text(json.dumps(CFG,ensure_ascii=False,indent=2),encoding="utf-8")
import shutil
for name in ["requirements.lock.txt","model_assets.json","install_spec.json"]:
    shutil.copy2(SETUP_DIR/name,RUN_DIR/name)

def run_stage(name):
    lock=RUN_DIR/"running.lock"
    try: fd=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY)
    except FileExistsError:
        raise RuntimeError(f"실행 잠금이 있습니다: {lock}. 다른 실행이 없는지 확인하세요. PC 강제 종료 후 남은 잠금만 수동 삭제하세요.")
    try:
        with os.fdopen(fd,"w") as f:f.write(str(os.getpid()))
        run_command([ENV_PYTHON,"-u",worker,RUN_DIR/"run_config.json",name],name+".log")
    finally: lock.unlink(missing_ok=True)

def show_result(name):
    p=RUN_DIR/(name+"_complete.json")
    if p.exists():
        r=json.loads(p.read_text(encoding="utf-8"))
        print("실제 파일 위치:",r["directory"])
        print(json.dumps(r["result"],ensure_ascii=False,indent=2))
    else: print("미완료:",name)
print("실험 ID:",EXPERIMENT_ID)
print("결과:",RUN_DIR)


실험 ID: TASK006-8f2c71505c11284c
결과: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006\TASK006-8f2c71505c11284c


## 6. 데이터 점검·고정 분할 실행
먼저 이 셀까지 실행하고 출력 개수·정답 분포·유사 이미지 후보를 확인하세요. 이상이 있으면 아래 GPU 단계로 넘어가지 말고 데이터를 점검하세요. 원본 CSV는 자동 수정하지 않습니다.

In [12]:
run_stage("audit")
show_result("audit")

image audit 250 / 6714
image audit 500 / 6714
image audit 750 / 6714
image audit 1000 / 6714
image audit 1250 / 6714
image audit 1500 / 6714
image audit 1750 / 6714
image audit 2000 / 6714
image audit 2250 / 6714
image audit 2500 / 6714
image audit 2750 / 6714
image audit 3000 / 6714
image audit 3250 / 6714
image audit 3500 / 6714
image audit 3750 / 6714
image audit 4000 / 6714
image audit 4250 / 6714
image audit 4500 / 6714
image audit 4750 / 6714
image audit 5000 / 6714
image audit 5250 / 6714
image audit 5500 / 6714
image audit 5750 / 6714
image audit 6000 / 6714
image audit 6250 / 6714
image audit 6500 / 6714
answer      a     b     c     d
split                          
holdout   259   242   255   244
pool     1064  1022  1075  1053
train     249   255   257   239
valid     128   125   129   118
{
  "csv_sha256": "83b31c210e42f9c47aa992960dfb5e5519cc2fdea089c82158ed2d73b1fa985b",
  "split_sha256": "c28b7ef0c5baf35689505b5bf4ad64168a754e118c0e4f59c0bf2b8d6884679d",
  "image_audit_

## 7. 소량 forward·generate·backward 검사
학습 집합의 첫 문항만 사용합니다. optimizer.step을 하지 않으며 다음 단계는 원본 모델을 새로 로드합니다. `supervised_tokens.csv`에서 실제 loss 대상 토큰을 확인하세요. 실패/OOM이면 자동으로 정밀도나 해상도를 바꾸지 않습니다.

In [13]:
run_stage("smoke")
show_result("smoke")

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 11:51:05.549000 17356 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<09:51,  1.28it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

## 8. 학습 전 모델 기준 평가
튜닝 검증 약 500개만 평가합니다. 한 글자 출력에 실패하면 오답으로 기록하며 fallback 사용률은 0입니다.

In [14]:
run_stage("base")
show_result("base")

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 11:51:38.913000 4432 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<09:03,  1.39it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: Fu

## 9. 새 LoRA 학습·저장·재로드 확인
고정 학습 약 1,000개로 1 epoch 학습합니다. 새 분할에서 처음부터 학습하며 기존 LoRA를 가져오지 않습니다. 검증 5개 이내의 저장 전/후 출력도 비교합니다.

**중단 후 동작:** 완료된 단계는 건너뜁니다. 학습 중간의 optimizer/RNG 재개는 구현하지 않았습니다. 학습 중 중단되면 부분 로그를 보존하고 **이 학습 단계만 처음부터** 다시 시작합니다.

In [15]:
run_stage("train")
show_result("train")

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 11:56:58.064000 4972 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:57,  1.41it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: Fu

## 10. 저장된 LoRA 기준 평가
새 프로세스에서 base와 저장된 어댑터를 다시 로드하고 동일 튜닝 검증셋을 평가합니다.

In [16]:
run_stage("lora_eval")
show_result("lora_eval")

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 13:10:49.377000 28768 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:59,  1.40it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

## 11. 학습 전/후 비교 및 오답 검토
Accuracy, 정답 수, 파싱 실패율, 시간, VRAM 및 개선/악화 문항을 저장합니다. 기존 학습 전 모델과 LoRA의 차이는 기준 기록이며, 특정 모델 채택을 자동 결정하지 않습니다.

In [17]:
run_stage("compare")
show_result("compare")

model   n  correct_n  accuracy  parse_failure_rate  fallback_usage_rate    seconds  seconds_per_sample  peak_allocated_gib  peak_reserved_gib
 base 500        417     0.834                 0.0                  0.0 293.212457            0.586425            7.547402          11.277344
 lora 500        417     0.834                 0.0                  0.0 299.204877            0.598410            7.601601          11.333984
{'gain': 18, 'loss': 18, 'accuracy_delta': 0.0, 'n': 500, 'adoption': 'pending; no automatic model selection', 'review_02': 'pending', 'holdout_evaluated': False, 'test_evaluated': False}
실제 파일 위치: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006\TASK006-8f2c71505c11284c\compare_20260922_131611_e6d00bfd
{
  "gain": 18,
  "loss": 18,
  "accuracy_delta": 0.0,
  "n": 500,
  "adoption": "pending; no automatic model selection",
  "review_02": "pending",
  "holdout_evaluated": false,
  "test_evaluated": false
}


## 결과 파일 찾기
각 `show_result()`의 **실제 파일 위치**에 결과가 있습니다.

| 단계 | 주요 파일 |
|---|---|
| audit | split_manifest.csv, train/valid/holdout/pool_ids.csv, image_audit.csv, near_duplicate_candidates.csv, data_findings.csv, 분포 CSV |
| smoke | smoke_metrics.json, supervised_tokens.csv, environment.json |
| base | valid_base_predictions.csv, valid_base_metrics.json |
| train | adapter_epoch1/, processor/, train_log.csv, train_updates.jsonl, train_metrics.json, reload_check.json |
| lora_eval | valid_lora_predictions.csv, valid_lora_metrics.json |
| compare | comparison.csv, paired_predictions.csv, changed_answers.csv, type_comparison.csv, error_review_template.csv, summary.json |
| compare 운영 기록 | PROJECT_STATUS_update.md, CHANGELOG.md — 공식 기준본 수정 없음 |

파싱 실패·최초 출력·최종 예측·정답을 문항별 기록합니다. 오류 분류 템플릿은 사람이 글자 인식/추론/출력/문항 오류로 검토해야 합니다.
**최종 검증셋은 평가하지 않습니다.** test.csv, dev.csv, submission.csv는 이 노트북 범위에 없습니다.

다음 단계는 같은 저장 체크포인트의 추론 해상도 384²/560²/784² 비교입니다. 여기서는 먼저 기준 평가까지만 수행합니다.
분할·평가 수정은 02 검토 대상이며 독립 검토 상태는 pending입니다. 인계문 작성·전달·모델 채택은 수행하지 않습니다.
